In [1]:
# cell_1_setup.py
!pip install -q transformers sentencepiece protobuf
!pip install -q datasets z3-solver spacy
!pip install -q tqdm scikit-learn evaluate pandas
!python -m spacy download en_core_web_sm

import os
folders = ['/kaggle/working/src', '/kaggle/working/data/proofwriter',
           '/kaggle/working/results', '/kaggle/working/prompts']
for f in folders:
    os.makedirs(f, exist_ok=True)
print("All libraries installed + folders created ✓")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 81.0 MB/s eta 0:00:0000:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
All libraries installed + folders created ✓


In [2]:
# cell_2_upload_src.py
import shutil, os

# Upload all src files via Kaggle sidebar → Add Data → Upload
# They appear in /kaggle/input/
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.endswith('.py'):
            src_path = os.path.join(root, file)
            shutil.copy(src_path, f'/kaggle/working/src/{file}')
            print(f"  ✓ {file}")

print("src files:", os.listdir('/kaggle/working/src'))

  ✓ symbolic_verifier.py
  ✓ llm_interface.py
  ✓ data_loader.py
  ✓ logic_translator.py
  ✓ config.py
  ✓ correction_loop.py
  ✓ evaluator.py
  ✓ __init__.py
src files: ['__pycache__', 'evaluator.py', 'llm_interface.py', 'symbolic_verifier.py', 'correction_loop.py', 'logic_translator.py', '__init__.py', 'config.py', 'data_loader.py']


In [3]:
# cell_3_upload_data.py
import shutil, os

for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file == 'test.jsonl':
            shutil.copy(os.path.join(root, file),
                       '/kaggle/working/data/proofwriter/test.jsonl')
            print("test.jsonl copied ✓")

print("Files:", os.listdir('/kaggle/working/data/proofwriter'))

test.jsonl copied ✓
Files: ['test.jsonl']


In [4]:
# cell_4_config.py
import os, sys
sys.path.insert(0, '/kaggle/working')

config_content = """
import os
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
QUANTIZE_4BIT = True
MAX_NEW_TOKENS = 128
TEMPERATURE = 0.3
MAX_RETRIES = 3
MAX_SAMPLES = 100
SKIP_UNTRANSLATABLE = True
ROOT = "/kaggle/working"
DATA_DIR = "/kaggle/working/data"
PROOFWRITER_DIR = "/kaggle/working/data/proofwriter"
CLUTRR_DIR = "/kaggle/working/data/clutrr"
RESULTS_DIR = "/kaggle/working/results"
PROMPTS_DIR = "/kaggle/working/prompts"
BASELINE_RESULTS = "/kaggle/working/results/baseline_results.json"
NEURO_SYM_RESULTS = "/kaggle/working/results/neuro_sym_results.json"
"""
with open('/kaggle/working/src/config.py', 'w') as f:
    f.write(config_content)
print("config.py ✓")

config.py ✓


In [5]:
# cell_5_llm_interface.py
llm_content = """
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
MAX_NEW_TOKENS = 128
TEMPERATURE = 0.3

_tokenizer = None
_model = None

def load_model(force_reload=False):
    global _tokenizer, _model
    if _tokenizer is not None and _model is not None and not force_reload:
        return _tokenizer, _model
    print(f"Loading model: {MODEL_ID}")
    _tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False)
    _tokenizer.pad_token = _tokenizer.eos_token
    _model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        device_map={"": "cuda:0"},
    )
    print(f"GPU used: {(torch.cuda.mem_get_info()[1]-torch.cuda.mem_get_info()[0])/1e9:.1f}GB")
    print("Model loaded ✓")
    return _tokenizer, _model

def _call_model(prompt, max_new_tokens=MAX_NEW_TOKENS):
    tokenizer, model = load_model()
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to("cuda:0") for k, v in inputs.items()}
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=TEMPERATURE,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

def generate_cot_steps(context, question):
    prompt = (
        "[INST] You are a precise logical reasoner. "
        "Given the facts and rules below, reason step by step.\\n\\n"
        "Rules for your output:\\n"
        "- Number each step exactly as: Step 1: ... Step 2: ...\\n"
        "- Each step must follow logically from previous steps and the facts.\\n"
        "- End with exactly: Answer: True / Answer: False / Answer: Unknown\\n\\n"
        f"Facts and Rules:\\n{context}\\n\\n"
        f"Question: {question}\\n[/INST]"
    )
    return _call_model(prompt)

def parse_steps(raw_output):
    steps = re.findall(
        r"Step\\s*\\d+:\\s*(.*?)(?=Step\\s*\\d+:|Answer:|$)",
        raw_output, re.DOTALL | re.IGNORECASE,
    )
    steps = [s.strip() for s in steps if s.strip()]
    answer_match = re.search(r"Answer:\\s*(True|False|Unknown)", raw_output, re.IGNORECASE)
    answer = answer_match.group(1).capitalize() if answer_match else None
    return steps, answer

def request_correction(context, question, accepted_steps, failed_step, reason):
    accepted_str = "\\n".join(f"Step {i+1}: {s}" for i, s in enumerate(accepted_steps)) or "(none yet)"
    prompt = (
        "[INST] A reasoning step you generated has a logical error. "
        "Fix ONLY that step.\\n\\n"
        f"Facts and Rules:\\n{context}\\n\\n"
        f"Question: {question}\\n\\n"
        f"Accepted steps so far:\\n{accepted_str}\\n\\n"
        f"PROBLEMATIC STEP: \\"{failed_step}\\"\\n"
        f"WHY IT FAILED: {reason}\\n\\n"
        "Write only the corrected step text:\\n[/INST]"
    )
    raw = _call_model(prompt, max_new_tokens=100)
    for line in raw.splitlines():
        line = line.strip()
        if line:
            return line
    return raw.strip()

def generate_final_answer(context, question, verified_steps):
    steps_str = "\\n".join(f"Step {i+1}: {s}" for i, s in enumerate(verified_steps))
    prompt = (
        "[INST] Based on the verified reasoning steps, "
        "give the final answer. Respond with ONLY: True, False, or Unknown.\\n\\n"
        f"Facts and Rules:\\n{context}\\n\\n"
        f"Question: {question}\\n\\n"
        f"Verified reasoning:\\n{steps_str}\\n\\n"
        "Final Answer: [/INST]"
    )
    raw = _call_model(prompt, max_new_tokens=20)
    match = re.search(r"\\b(True|False|Unknown)\\b", raw, re.IGNORECASE)
    return match.group(1).capitalize() if match else raw.strip()
"""
with open('/kaggle/working/src/llm_interface.py', 'w') as f:
    f.write(llm_content)
print("llm_interface.py ✓")

llm_interface.py ✓


In [6]:
# cell_6_data_loader.py
data_loader_content = """
import json
import os
from src.config import PROOFWRITER_DIR, MAX_SAMPLES


def load_proofwriter(split="test", max_samples=MAX_SAMPLES):
    path = os.path.join(PROOFWRITER_DIR, f"{split}.jsonl")
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"ProofWriter data not found at {path}.\\n"
            "Download from https://allenai.org/data/proofwriter"
        )

    samples = []
    with open(path) as f:
        for line in f:
            if not line.strip():
                continue
            entry = json.loads(line.strip())

            structured_facts = []
            for key, val in entry.get('triples', {}).items():
                structured_facts.append({
                    'text':           val['text'],
                    'representation': val['representation']
                })

            structured_rules = []
            for key, val in entry.get('rules', {}).items():
                structured_rules.append({
                    'text':           val['text'],
                    'representation': val['representation']
                })

            context = entry['theory']

            for q_key, q_val in entry['questions'].items():
                raw_answer = q_val['answer']
                if raw_answer is True:
                    answer = "True"
                elif raw_answer is False:
                    answer = "False"
                else:
                    answer = str(raw_answer)

                samples.append({
                    'id':               f"{entry['id']}_{q_key}",
                    'context':          context,
                    'question':         q_val['question'],
                    'answer':           answer,
                    'structured_facts': structured_facts,
                    'structured_rules': structured_rules,
                    'dataset':          'proofwriter',
                })

                if len(samples) >= max_samples:
                    return samples

    return samples


def load_dataset_by_name(name, split="test", max_samples=MAX_SAMPLES):
    loaders = {
        "proofwriter": lambda: load_proofwriter(split, max_samples),
    }
    if name not in loaders:
        raise ValueError(f"Unknown dataset: {name}")
    return loaders[name]()
"""
with open('/kaggle/working/src/data_loader.py', 'w') as f:
    f.write(data_loader_content)
print("data_loader.py ✓")

data_loader.py ✓


In [7]:
# cell_7_core_files.py
import sys, os
os.makedirs('/kaggle/working/src', exist_ok=True)
for key in list(sys.modules.keys()):
    if key.startswith('src'):
        del sys.modules[key]

with open('/kaggle/working/src/logic_translator.py', 'w') as f:
    f.write('''import re
from z3 import Bool, Implies, Not, And

class NLToLogicTranslator:
    def __init__(self):
        self.predicates = {}
        self.failed_translations = []

    def _get(self, subject, relation, obj):
        key = f"{subject}_{relation}_{obj}".lower()
        key = re.sub(r"[^a-z0-9_]", "", re.sub(r"\\s+", "_", key))
        key = re.sub(r"_+", "_", key).strip("_") or "unknown"
        if key not in self.predicates:
            self.predicates[key] = Bool(key)
        return self.predicates[key]

    def _parse_triple(self, s):
        m = re.match(r\'\\(\\ *"([^"]+)"\\ +"([^"]+)"\\ +"([^"]+)"\\ +"([+-])"\\ *\\)\', s)
        if not m: return None
        subj, rel, obj, sign = m.groups()
        p = self._get(subj, rel, obj)
        return Not(p) if sign == "-" else p

    def _find_triples(self, text):
        pat = r\'\\(\\ *"[^"]+"\\ +"[^"]+"\\ +"[^"]+"\\ +"[+-]"\\ *\\)\'
        return [e for e in [self._parse_triple(m) for m in re.findall(pat, text)] if e is not None]

    def parse_representation(self, rep):
        rep = rep.strip()
        if "->" not in rep:
            return self._parse_triple(rep)
        ant, con = rep.split("->", 1)
        ae = self._find_triples(ant)
        ce = self._find_triples(con)
        if not ae or not ce: return None
        return Implies(And(*ae) if len(ae) > 1 else ae[0], ce[0])

    def load_structured_context(self, facts, rules):
        exprs = []
        for x in facts + rules:
            e = self.parse_representation(x["representation"])
            if e is not None: exprs.append(e)
        return exprs

    def translate(self, sentence):
        s = sentence.strip().rstrip(".")
        try:
            m = re.match(r"[Ii]f\\s+(.+?)\\s+then\\s+(.+)", s)
            if m:
                a = self._nl_atom(m.group(1))
                b = self._nl_atom(m.group(2))
                if a and b: return Implies(a, b)
            return self._nl_atom(s)
        except: return None

    def _is_neg(self, t):
        return bool(re.search(r"\\b(not|is not|does not|never)\\b", t, re.I))

    def _strip_neg(self, t):
        t = re.sub(r"\\bis not\\b", "is", t, flags=re.I)
        t = re.sub(r"\\bnot\\b", "", t, flags=re.I)
        return re.sub(r"\\s+", " ", t).strip()

    def _nl_atom(self, text):
        text = text.strip().rstrip(".")
        if self._is_neg(text):
            clean = self._strip_neg(text)
            key = re.sub(r"[^a-z0-9_]", "", re.sub(r"\\s+", "_", clean.lower())).strip("_") or "unknown"
            if key not in self.predicates: self.predicates[key] = Bool(key)
            return Not(self.predicates[key])
        key = re.sub(r"[^a-z0-9_]", "", re.sub(r"\\s+", "_", text.lower())).strip("_") or "unknown"
        if key not in self.predicates: self.predicates[key] = Bool(key)
        return self.predicates[key]

    def translate_context(self, context_text):
        return [e for s in re.split(r"(?<=[.!?])\\s+", context_text)
                for e in [self.translate(s)] if e is not None]

    def report(self):
        print(f"[Translator] {len(self.predicates)} predicates")
''')
print("logic_translator.py ✓")

with open('/kaggle/working/src/symbolic_verifier.py', 'w') as f:
    f.write('''from z3 import Solver, unsat
from src.logic_translator import NLToLogicTranslator

class SymbolicVerifier:
    def __init__(self):
        self.solver = Solver()
        self.translator = NLToLogicTranslator()
        self.accepted_steps = []
        self.skipped_steps = []
        self.rejected_steps = []

    def add_context(self, context_text, structured_facts=None, structured_rules=None):
        if structured_facts and structured_rules:
            exprs = self.translator.load_structured_context(structured_facts, structured_rules)
            for expr in exprs:
                self.solver.add(expr)
            print(f"  [Z3] Loaded {len(exprs)} structured expressions")

    def verify_step(self, step_text):
        expr = self.translator.translate(step_text)
        if expr is None:
            self.skipped_steps.append(step_text)
            return True, "skip"
        self.solver.push()
        self.solver.add(expr)
        result = self.solver.check()
        self.solver.pop()
        if result == unsat:
            self.rejected_steps.append(step_text)
            return False, "contradiction"
        self.solver.add(expr)
        self.accepted_steps.append(step_text)
        return True, "valid"

    def reset(self):
        self.solver = Solver()
        self.translator = NLToLogicTranslator()
        self.accepted_steps = []
        self.skipped_steps = []
        self.rejected_steps = []

    def summary(self):
        return {"accepted": len(self.accepted_steps),
                "skipped": len(self.skipped_steps),
                "rejected": len(self.rejected_steps)}
''')
print("symbolic_verifier.py ✓")

with open('/kaggle/working/src/correction_loop.py', 'w') as f:
    f.write('''from src.llm_interface import generate_cot_steps, parse_steps, request_correction, generate_final_answer
from src.symbolic_verifier import SymbolicVerifier

MAX_RETRIES = 2

def run_pipeline(sample, verbose=False):
    verifier = SymbolicVerifier()
    verifier.add_context(
        sample["context"],
        structured_facts=sample.get("structured_facts"),
        structured_rules=sample.get("structured_rules")
    )
    raw = generate_cot_steps(sample["context"], sample["question"])
    steps, _ = parse_steps(raw)
    final_steps = []
    corrections = []
    for idx, step in enumerate(steps):
        is_valid, reason = verifier.verify_step(step)
        if verbose:
            print(f"  Step {idx+1} {"✓" if is_valid else "✗"}: {step[:60]}")
        if not is_valid:
            for attempt in range(1, MAX_RETRIES + 1):
                candidate = request_correction(
                    sample["context"], sample["question"], final_steps, step, reason)
                ok, _ = verifier.verify_step(candidate)
                if ok:
                    corrections.append({"step_idx": idx, "original": step, "corrected": candidate})
                    step = candidate
                    break
        final_steps.append(step)
    final_answer = generate_final_answer(sample["context"], sample["question"], final_steps)
    ground_truth = str(sample.get("answer", "")).capitalize()
    return {
        "id": sample.get("id", -1),
        "steps": final_steps,
        "answer": final_answer,
        "ground_truth": ground_truth,
        "correct": final_answer.lower() == ground_truth.lower(),
        "corrections": corrections,
        "num_corrections": len(corrections),
        "verifier_summary": verifier.summary(),
    }
''')
print("correction_loop.py ✓")

logic_translator.py ✓
symbolic_verifier.py ✓
correction_loop.py ✓


In [12]:
# cell_8_test_structured_parsing.py
import sys
for key in list(sys.modules.keys()):
    if key.startswith('src'):
        del sys.modules[key]

from src.logic_translator import NLToLogicTranslator
import json

t = NLToLogicTranslator()

with open('/kaggle/working/data/proofwriter/test.jsonl') as f:
    entry = json.loads(f.readline())

facts = [{'text': v['text'], 'representation': v['representation']}
         for v in entry['triples'].values()]
rules = [{'text': v['text'], 'representation': v['representation']}
         for v in entry['rules'].values()]

print("=== FACTS ===")
for fact in facts[:3]:
    print(f"Text: {fact['text']}")
    expr = t.parse_representation(fact['representation'])
    print(f"Z3:   {expr}")
    print()

print("=== RULES ===")
for rule in rules[:2]:
    print(f"Text: {rule['text']}")
    expr = t.parse_representation(rule['representation'])
    print(f"Z3:   {expr}")
    print()

t.report()
print()
print("If all facts and rules show Z3 expressions above → Option A working ✓")

=== FACTS ===
Text: The cow is not big.
Z3:   Not(cow_is_big)

Text: The cow is not green.
Z3:   Not(cow_is_green)

Text: The lion eats the tiger.
Z3:   lion_eats_tiger

=== RULES ===
Text: If something sees the squirrel and the squirrel eats the cow then the cow is round.
Z3:   Implies(And(something_sees_squirrel, squirrel_eats_cow),
        cow_is_round)

Text: If something is green then it eats the tiger.
Z3:   Implies(something_is_green, something_eats_tiger)

[Translator] 8 predicates

If all facts and rules show Z3 expressions above → Option A working ✓


In [8]:
# cell_9_load_model.py
import sys
for key in list(sys.modules.keys()):
    if key.startswith('src'):
        del sys.modules[key]

from src.llm_interface import load_model
load_model()

import torch
print(f"GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f}GB")
print("LLM ready ✓")

Loading model: mistralai/Mistral-7B-Instruct-v0.2


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

GPU used: 14.6GB
Model loaded ✓
GPU free: 1.0GB
LLM ready ✓


In [9]:
# cell_10_upload_baseline.py
import shutil, os

os.makedirs('/kaggle/working/results', exist_ok=True)

for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file == 'baseline_results.json':
            shutil.copy(os.path.join(root, file),
                       '/kaggle/working/results/baseline_results.json')
            print("baseline_results.json restored ✓")

print("Results folder:", os.listdir('/kaggle/working/results'))

baseline_results.json restored ✓
Results folder: ['baseline_results.json']


In [10]:
# cell_11_neuro_sym_pipeline.py
import torch, gc, os, sys
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from tqdm import tqdm
from src.correction_loop import run_pipeline
from src.data_loader import load_proofwriter
from src.evaluator import compute_metrics, save_results, load_results, print_comparison

# Confirm model in memory
import src.llm_interface as llm_mod
if llm_mod._model is None:
    from src.llm_interface import load_model
    load_model()
print(f"GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f}GB")

samples = load_proofwriter(max_samples=100)
print(f"Loaded {len(samples)} samples")

results = []
correct_count = 0
total_corrections = 0

for i, sample in enumerate(tqdm(samples, desc="Neuro-Sym")):
    try:
        result = run_pipeline(sample, verbose=False)
        results.append(result)
        if result['correct']: correct_count += 1
        total_corrections += result['num_corrections']
    except Exception as e:
        print(f"\n  Error {i}: {str(e)[:80]}")
        gc.collect()
        torch.cuda.empty_cache()
        results.append({
            'id': sample['id'], 'steps': [], 'answer': None,
            'ground_truth': sample['answer'], 'correct': False,
            'corrections': [], 'num_corrections': 0, 'verifier_summary': {}
        })
        continue
    if (i+1) % 10 == 0:
        print(f"\n  [{i+1}/100] Acc: {correct_count/(i+1):.2%} | Corr: {total_corrections/(i+1):.2f}")
        gc.collect()
        torch.cuda.empty_cache()

metrics = compute_metrics(results)
save_results(results, metrics, '/kaggle/working/results/neuro_sym_results.json')
print(f"\nAccuracy: {metrics['accuracy']:.2%}")
print(f"Logical consistency: {metrics['logical_consistency_rate']:.2%}")
print(f"Avg corrections: {metrics['avg_corrections_per_sample']:.2f}")

_, bm = load_results('/kaggle/working/results/baseline_results.json')
print_comparison(bm, metrics)
print("Done ✓")

GPU free: 1.0GB
Loaded 100 samples


Neuro-Sym:   0%|          | 0/100 [00:00<?, ?it/s]

  [Z3] Loaded 20 structured expressions


Neuro-Sym:   1%|          | 1/100 [00:02<04:50,  2.93s/it]

  [Z3] Loaded 20 structured expressions


...

  [Z3] Loaded 21 structured expressions


Neuro-Sym:  99%|█████████▉| 99/100 [37:49<00:27, 27.72s/it]

  [Z3] Loaded 21 structured expressions


Neuro-Sym: 100%|██████████| 100/100 [37:55<00:00, 22.76s/it]


  [100/100] Acc: 22.00% | Corr: 0.00
Results saved → /kaggle/working/results/neuro_sym_results.json

Accuracy: 22.00%
Logical consistency: 100.00%
Avg corrections: 0.00

  METRIC                                BASELINE  NEURO-SYM    DELTA
  accuracy                                0.3400     0.2200  -0.1200
  logical_consistency_rate                1.0000     1.0000  +0.0000
  avg_corrections_per_sample              0.0000     0.0000  +0.0000

Done ✓


In [11]:
# cell_12_save_results.py
import os, json

for f in os.listdir('/kaggle/working/results'):
    size = os.path.getsize(f'/kaggle/working/results/{f}')
    with open(f'/kaggle/working/results/{f}') as rf:
        data = json.load(rf)
    metrics = data.get('metrics', {})
    print(f"{f}")
    print(f"  Samples:     {metrics.get('total_samples', 'N/A')}")
    print(f"  Accuracy:    {metrics.get('accuracy', 0):.2%}")
    print(f"  Size:        {size/1024:.1f} KB")
    print()

print("Results are in Output tab → download from there ✓")

neuro_sym_results.json
  Samples:     100
  Accuracy:    22.00%
  Size:        63.3 KB

baseline_results.json
  Samples:     100
  Accuracy:    34.00%
  Size:        64.5 KB

Results are in Output tab → download from there ✓
